# Bronze to Silver Transformation - Device Logs

**Pipeline Stage**: Data Quality & Enrichment  
**Source**: Bronze Layer - 'Device_logs'

**Target**: Silver Layer - Cleaned, validated, and enriched device performance logs

---

In [0]:
device_logs_df = spark.read.format("delta").load('/Volumes/telecom_catalog/default/bronze/Device_logs/')

## 1. Data Exploration
Initial exploration of schema and data quality.

**Objective**: Load raw device logs from Bronze layer and perform initial data profiling

**Key Activities**:
* Load Delta table from Bronze layer
* Display sample records to understand data structure
* Examine schema and data types
* Identify duplicates and null values
* Calculate basic statistics

In [0]:
device_logs_df.display()

In [0]:
device_logs_df.printSchema()

In [0]:
from pyspark.sql.functions import *

print("Total Rows:", device_logs_df.count())
print("Duplicate Rows:", device_logs_df.count() - device_logs_df.dropDuplicates().count())

# Check null values across all columns
display(
    device_logs_df.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in device_logs_df.columns
    ])
)

## 2. Data Quality Checks
Identify duplicates, null values, and data anomalies.

**Objective**: Detect and handle duplicate records in device logs

**Business Logic**:
* Duplicates are identified by `(device_id, event_time)` combination
* When duplicates exist, keep the record with the highest `offset` value (latest ingested)
* Use SQL window function `ROW_NUMBER()` to rank and deduplicate

In [0]:
device_logs_df.createOrReplaceTempView("device_logs")

silver_df = spark.sql("""
SELECT * EXCEPT(rn)
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY device_id, event_time
               ORDER BY offset DESC
           ) AS rn
    FROM device_logs
)
WHERE rn = 1
""")

## 3. Data Cleaning & Standardization
Standardize business keys and handle missing values.

**Objective**: Ensure consistent data formats and handle missing values appropriately

**Transformations**:
* **device_id**: UPPER case (e.g., "dvc_123" → "DVC_123")
* **device_type**: lower case (e.g., "ROUTER" → "router")
* **region**: Title case (e.g., "north america" → "North America")
* **status**: lower case (e.g., "ACTIVE" → "active")
* **temp**: Replace NULL with -1 (indicates missing sensor)

In [0]:
from pyspark.sql.functions import col, trim, upper, lower, initcap

silver_df = (
    silver_df
    .withColumn("device_id", upper(trim(col("device_id"))))
    .withColumn("device_type", lower(trim(col("device_type"))))
    .withColumn("region", initcap(trim(col("region"))))
    .withColumn("status", lower(trim(col("status"))))
)

In [0]:
from pyspark.sql.functions import col, when

silver_df = silver_df.withColumn(
    "temp",
    when(col("temp").isNull(), -1)
    .otherwise(col("temp"))
)

## 4. Business Rules & Validation
Define valid ranges and create validation flags.

**Objective**: Validate metrics against business rules and flag invalid records

**Validation Rules**:
* **cpu_usage**: Must be between 0-100%
* **memory_usage**: Must be between 0-100%
* **packet_loss**: Must be >= 0
* **latency_ms**: Must be >= 0
* **temp**: Either -1 (missing sensor) or between 0-100°C

**Outputs**:
* `validation_reason`: Describes why a record is invalid (or "Valid")
* `is_valid`: Boolean flag for downstream filtering

In [0]:
from pyspark.sql.functions import when, col

silver_df = silver_df.withColumn(
    "validation_reason",
    when(col("cpu_usage").isNull(), "CPU Missing")
    .when(~col("cpu_usage").between(0,100), "Invalid CPU")
    .when(~col("memory_usage").between(0,100), "Invalid Memory")
    .when(col("packet_loss") < 0, "Invalid Packet Loss")
    .when(col("latency_ms") < 0, "Invalid Latency")
    .when(~((col("temp")==-1) | (col("temp").between(0,100))), "Invalid Temperature")
    .otherwise("Valid")
)

In [0]:
from pyspark.sql.functions import when, col

silver_df = silver_df.withColumn(
    "is_valid",
    when(col("validation_reason")=="Valid", True)
    .otherwise(False)
)

## 5. Feature Engineering
Create derived business metrics for downstream analytics.

**Objective**: Create business-friendly categorizations and health indicators

**Derived Metrics**:
* **cpu_status**: Normal (<60%) | High (60-85%) | Critical (>85%)
* **memory_status**: Normal (<60%) | High (60-85%) | Critical (>85%)
* **network_health**: Healthy (≤2% loss) | Moderate (2-5%) | Poor (>5%)
* **latency_category**: Low (<100ms) | Medium (100-300ms) | High (>300ms)
* **temperature_status**: Normal (<60°C) | High (60-80°C) | Critical (>80°C)
* **overall_health_score**: Excellent | Good | Needs Attention

These features enable business users to quickly identify problematic devices.

In [0]:
from pyspark.sql.functions import when, col

silver_df = silver_df.withColumn(
    "cpu_status",
    when(col("cpu_usage").isNull(), "Unknown")
    .when(col("cpu_usage") < 60, "Normal")
    .when(col("cpu_usage") < 85, "High")
    .when(col("cpu_usage") <= 100, "Critical")
    .otherwise("Invalid"))

In [0]:
from pyspark.sql.functions import when, col

silver_df = silver_df.withColumn(
    "memory_status",
    when(col("memory_usage") < 60, "Normal")
    .when(col("memory_usage") < 85, "High")
    .otherwise("Critical")
)

In [0]:
from pyspark.sql.functions import when, col

silver_df = silver_df.withColumn(
    "network_health",
    when(col("packet_loss") <= 2, "Healthy")
    .when(col("packet_loss") <= 5, "Moderate")
    .otherwise("Poor")
)

In [0]:
from pyspark.sql.functions import when, col

silver_df = silver_df.withColumn(
    "latency_category",
    when(col("latency_ms") < 100, "Low")
    .when(col("latency_ms") < 300, "Medium")
    .otherwise("High")
)

In [0]:
from pyspark.sql.functions import when, col

silver_df = silver_df.withColumn(
    "temperature_status",
    when(col("temp") == -1, "Sensor Missing")
    .when(col("temp") < 60, "Normal")
    .when(col("temp") < 80, "High")
    .otherwise("Critical")
)

In [0]:
from pyspark.sql.functions import when, col

silver_df = silver_df.withColumn(
    "overall_health_score",
    when(
        (col("cpu_status") == "Normal") &
        (col("memory_status") == "Normal") &
        (col("network_health") == "Healthy"),
        "Excellent"
    ).when(
        (col("cpu_status").isin("Normal", "High")) &
        (col("memory_status").isin("Normal", "High")),
        "Good"
    ).otherwise("Needs Attention")
)

## 6. Audit Columns
Add metadata for data lineage and troubleshooting.

**Objective**: Add tracking columns for data governance and debugging

**Audit Fields**:
* **Device_logs_load_time**: Timestamp when record was processed into Silver layer

These fields help:
* Track data freshness
* Debug transformation issues
* Support incremental processing patterns

In [0]:
from pyspark.sql.functions import current_timestamp

silver_df = silver_df.withColumn(
    "Device_logs_load_time",
    current_timestamp()
)

## 7. Write to Silver Layer
Persist the cleaned and enriched data to the silver layer.

**Objective**: Save the transformed data to Unity Catalog Silver schema

**Configuration**:
* **Format**: Delta Lake
* **Mode**: Overwrite (full refresh)
* **Target**: `telecom_catalog.silver_schema.Device_logs`
* **Options**: `overwriteSchema=True` (allows schema evolution)

**Post-Write Validation**: Query the table to verify successful load and data quality.

In [0]:
(
    silver_df.write.format("delta")
    .mode('overwrite')
    .option('overwriteSchema', True)
    .saveAsTable('telecom_catalog.silver_schema.Device_logs')
)